# CuTeDSL 03: 逐元素 Kernel 实现

在本教程中，你将逐步学习如何使用 CuTe DSL 构建高效的逐元素 kernel：
- 如何使用 CuTe DSL 结合基本 CUDA 技术实现 GPU kernel
- 如何对 kernel 进行性能基准测试
- 如何对 tensor 进行分块和分区，并映射到基本 CuTe 布局
- 什么是 Thread & Value 布局以及从 thread 和 value 索引到逻辑坐标的映射
- 如何使用 TV 布局实现高级 kernel 并调优性能以达到峰值性能
- 使用 benchmark 工具测量性能
- 使用 autotune 工具自动调优参数

In [1]:
import torch
from functools import partial
from typing import List

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack


## 一、理解逐元素加法

逐元素加法是线性代数和深度学习中的基本操作。给定两个形状相同的 tensor，该操作对元素进行逐个相加，生成相同形状的结果 tensor。

对于两个形状为 $(M, N)$ 的二维 tensor $A$ 和 $B$，逐元素加法操作 $C = A + B$ 定义为：

$
   C_{i,j} = A_{i,j} + B_{i,j}
$

其中：
- $i \in [0, M-1]$ 表示行索引
- $j \in [0, N-1]$ 表示列索引
- $A_{i,j}$、$B_{i,j}$ 和 $C_{i,j}$ 分别是 tensor $A$、$B$ 和 $C$ 在位置 $(i,j)$ 处的元素

该操作有几个重要特性：
1. **可并行化**：每个元素可以独立计算
2. **内存受限**：性能受内存带宽限制，而非计算能力
3. **对合并访问敏感**：效率取决于内存访问模式
4. **适合向量化**：多个元素可以一起处理

## 二、朴素逐元素加法 Kernel

让我们从一个朴素实现开始，建立基准线，然后再探索优化方案。

### 朴素 Kernel 代码

In [2]:
# 基本 Kernel 实现, 采用 thread 和 tensor 元素之间简单的 1:1 映射。
@cute.kernel
def naive_elementwise_add_kernel(
    gA: cute.Tensor,  # 输入 tensor A
    gB: cute.Tensor,  # 输入 tensor B
    gC: cute.Tensor,  # 输出 tensor C = A + B
):
    # 步骤 1：获取 thread 索引
    tidx, _, _ = cute.arch.thread_idx()  # block 内的 thread 索引（0 到 bdim-1）
    bidx, _, _ = cute.arch.block_idx()  # grid 中的 block 索引（0 到 grid_dim-1）
    bdim, _, _ = cute.arch.block_dim()  # 每个 block 的 thread 数量

    # 计算全局 thread 索引
    thread_idx = bidx * bdim + tidx  # 全局 thread ID

    # 步骤 2：将 thread 索引映射到 tensor 坐标
    m, n = gA.shape  # 获取 tensor 维度（M 行 x N 列）, 每个 thread 将处理输入 tensor 的一个元素

    # 将线性 thread 索引转换为 2D 坐标：
    mi = thread_idx // n  # 行索引（慢速变化的维度）（0 到 m-1）
    ni = thread_idx % n  # 列索引（快速变化的维度）（0 到 n-1）

    # 步骤 3：加载和处理数据
    # 从输入 tensor 加载值, tensor 布局自动处理从逻辑索引 (mi, ni) 到物理内存地址的转换
    a_val = gA[mi, ni]  # 从 tensor A 加载元素
    b_val = gB[mi, ni]  # 从 tensor B 加载元素

    # 步骤 4：存储结果
    # 将和写回输出 tensor
    gC[mi, ni] = a_val + b_val

### Kernel 启动配置和测试

本节演示如何：
1. 使用 `cute.jit` 函数配置和启动 kernel
2. 使用 `torch` 设置测试数据
3. 验证正确性

**启动配置**
  - 每个 block 使用 256 个 thread（常见选择，可获得良好的占用率）
  - 根据总元素数计算 grid 大小：`(m * n) // threads_per_block`
  - 为简单起见使用单维度 block 和 grid 配置

首先编写用于启动 kernel 的 Host JIT 函数

In [3]:
@cute.jit  # 即时编译装饰器
def naive_elementwise_add(
    mA: cute.Tensor,  # 输入 tensor A
    mB: cute.Tensor,  # 输入 tensor B
    mC: cute.Tensor,  # 输出 tensor C
):
    # 配置 kernel 启动参数
    # --------------------------------
    # 选择每个 block 的 thread 数量
    # 256 是常见选择，因为它：
    # - 在大多数 GPU 上允许良好的占用率
    # - 是 32（warp 大小）的倍数
    # - 提供足够的 thread 来隐藏延迟
    num_threads_per_block = 256

    # 获取输入维度
    m, n = mA.shape  # 矩阵维度（M 行 x N 列）

    # 创建 kernel 实例
    kernel = naive_elementwise_add_kernel(mA, mB, mC)

    # 使用计算的 grid 维度启动 kernel
    # -------------------------------------------
    # Grid 大小计算：
    # - 总元素数：m * n
    # - 需要的 block 数：ceil(total_elements / threads_per_block)
    # - 这里使用整数除法假设 m * n 是 threads_per_block 的倍数
    kernel.launch(
        grid=((m * n) // num_threads_per_block, 1, 1),  # x,y,z 方向的 block 数量
        block=(num_threads_per_block, 1, 1),  # x,y,z 方向每个 block 的 thread 数
    )

使用 torch 设置测试数据

In [4]:
# 测试设置
# ----------
# 定义测试维度
M, N = 16384, 8192  # 使用大矩阵来测量性能

# 在 GPU 上创建测试数据
# 使用 float16（半精度）是因为：
# - 减少内存带宽需求
# - 在现代 GPU 上性能更好
a = torch.randn(M, N, device="cuda", dtype=torch.float16)  # 随机输入 A
b = torch.randn(M, N, device="cuda", dtype=torch.float16)  # 随机输入 B
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)  # 输出缓冲区

# 计算总元素数用于带宽计算
num_elements = sum([a.numel(), b.numel(), c.numel()])

# 将 PyTorch tensor 转换为 CuTe tensor
# -------------------------------------
# from_dlpack 创建 PyTorch tensor 的 CuTe tensor 视图
# assumed_align=16 确保向量化访问的正确内存对齐
a_ = from_dlpack(a, assumed_align=16)  # CuTe tensor A
b_ = from_dlpack(b, assumed_align=16)  # CuTe tensor B
c_ = from_dlpack(c, assumed_align=16)  # CuTe tensor C

编译和运行

In [5]:
# 为特定输入类型编译 kernel
naive_elementwise_add_ = cute.compile(naive_elementwise_add, a_, b_, c_)

# 运行 kernel
naive_elementwise_add_(a_, b_, c_)

# 验证结果
# -------------
# 将我们的 kernel 输出与 PyTorch 的原生实现进行比较
torch.testing.assert_close(c, a + b)  # 如果结果不匹配则抛出错误

## 三、性能分析和基准测试

为了理解和改进 kernel 的性能，我们需要测量其执行时间和内存吞吐量。让我们分析几个关键指标：

* **执行时间**
   - 以微秒为单位测量原始 kernel 性能，越低越好
   - 受 GPU 时钟速度、内存带宽和 kernel 效率影响
* **内存吞吐量**
   - 测量数据复制速度（GB/s），越高越好
   - 理论峰值因 GPU 型号而异，对于逐元素加法：
     * 读取：2 个元素（A 和 B），写入：1 个元素（C）
     * 总字节数 = (2 次读取 + 1 次写入) x 元素数 x sizeof(dtype)

### 基准测试工具
以下是我们测量这些指标的基准测试工具：

In [6]:
def benchmark(callable, a_, b_, c_):
    avg_time_us = cute.testing.benchmark(
        callable,
        kernel_arguments=cute.testing.JitArguments(a_, b_, c_),
        warmup_iterations=5,
        iterations=100,
    )

    # 计算指标
    # ----------------
    dtype = a_.element_type

    # 计算传输的总字节数：
    # - 2 次读取（A 和 B）+ 1 次写入（C）
    # - 每个元素是 dtype.width 位
    bytes_per_element = dtype.width // 8
    total_bytes = num_elements * bytes_per_element

    # 计算实际带宽
    achieved_bandwidth = total_bytes / (avg_time_us * 1000)  # GB/s

    # 打印结果
    # ------------
    print(f"性能指标:")
    print(f"-------------------")
    print(f"Kernel 执行时间: {avg_time_us:.4f} us")
    print(f"内存吞吐量: {achieved_bandwidth:.2f} GB/s")

In [7]:
benchmark(naive_elementwise_add_, a_, b_, c_)

性能指标:
-------------------
Kernel 执行时间: 399.4547 us
内存吞吐量: 2016.01 GB/s


### 理论分析

本节通过几个理论框架分析逐元素加法 kernel 的性能特征和优化机会。

Little 定律提供了关于延迟、带宽和在途操作之间关系的重要见解：

$ L = \lambda \times W $

其中：
- $L$：需要的在途内存操作数量
- $\lambda$：目标内存带宽（字节/周期）
- $W$：内存系统延迟（周期）

根据 *Little 定律*，朴素实现具有
   - 每个 thread 1 个元素（4 字节加载 + 2 字节存储）
   - 256 threads/block x N 个 block
   - 有限的在途操作

在某些 GPU 上，这种并行度不足以饱和内存带宽。

基于这个分析，一种常用技术是**向量化**。向量化允许每次加载多个元素，而不是每个 thread 每次加载 1 个元素
   - 减少指令数量
   - 改善内存合并
   - 增加在途操作数

## 四、向量化加载和存储

为了根据 Little 定律提高性能，我们需要增加在途请求的数量。我们可以通过向量化内存访问来增加每个 thread 每次加载和存储操作处理的字节数。

由于 Ampere GPU 支持每次加载/存储最多 128 位，而每个元素是 16 位，我们可以在连续行上每次向量化操作加载 8 个元素。CuTe 分块操作使这种向量化变得简单直接。

使用 ``tiled_tensor = cute.zipped_divide(tensor, tiler)``，我们可以将输入 ``tensor`` 分成 ``tiler`` 块的组。对于向量化，我们将 ``tiler`` 指定为每个 thread 访问的数据块（同一行中的 8 个连续元素，即 ``(1,8)``）。然后不同的 thread 可以通过索引 ``tiled_tensor`` 的第二个模式来访问不同的块。

```python
mA : cute.Tensor                           # (2048,2048):(2048,1)
gA = cute.zipped_divide(a, tiler=(1, 8))   # 分块/向量化 => ((1,8),(2048,256)):((0,1),(2048,8))
```

$
    \begin{array}{ccccc}
    & ((1,8) & , & (2048,256)) & : ((0,1),(2048,8)) \\
    & \underbrace{\phantom{(1,8)}}_{tiler} & & \underbrace{\phantom{(2048,256)}}_{threads} & \\
    & \text{\scriptsize 块内} & & \text{\scriptsize 块数}
    \end{array}
$

```
层级                   shape	    stride	   含义
第一个模式 (1,8)	    tiler 本身	  (0,1)	    单个块内部的坐标（每个 thread 处理的数据）
第二个模式 (2048,256)	块的数量	  (2048,8)	 哪个块/哪个thread

想象把 2048x2048 的矩阵切成很多 1x8 的小条：
M 维度：2048 / 1 = 2048 个块（行方向不分块）
N 维度：2048 / 8 = 256 个块（列方向每 8 个元素一组）
所以一共有 2048 x 256 = 524,288 个小块，每个小块是连续的 8 个元素。
```

**stride 的对应关系**

第一个模式（块内部） (1,8) : (0,1)
- M 维度 stride = 0：块内只有 1 行，M 方向不需要移动
- N 维度 stride = 1：块内 8 个元素在内存中连续

第二个模式（块之间） (2048,256) : (2048,8)
- M 维度 stride = 2048：跨行移动 2048 个元素（和原始矩阵的行 stride 一致）
- N 维度 stride = 8：相邻块之间间隔 8 个元素（因为每个块占 8 个元素）


In [8]:
@cute.kernel
def vectorized_elementwise_add_kernel(
    gA: cute.Tensor,
    gB: cute.Tensor,
    gC: cute.Tensor,
):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()

    thread_idx = bidx * bdim + tidx

    # 以向量为单位将 thread 索引映射到输入 tensor 的逻辑索引
    m, n = gA.shape[1]  # thread 域，第二个模式的 shape = (2048, 256)，即 m=2048, n=256
    mi = thread_idx // n
    ni = thread_idx % n

    # 通过 tensor 布局将逻辑索引映射到物理地址
    a_val = gA[(None, (mi, ni))].load()
    b_val = gB[(None, (mi, ni))].load()
    print(f"[DSL INFO] sliced gA = {gA[(None, (mi, ni))]}")
    print(f"[DSL INFO] sliced gB = {gB[(None, (mi, ni))]}")

    # 执行逐元素加法
    gC[(None, (mi, ni))] = a_val + b_val

这个向量化 kernel 遵循与其朴素非向量化版本类似的结构，但有一个关键区别：tensor 切片模式。通过使用 `(None, (mi, ni))` 作为切片索引，我们可以从 `gA`、`gB` 和 `gC` 中提取一个 `(1,8)` 子 tensor，实现每个 thread 一次加载 8 个元素，如下所示

$ gA[(None, (mi, ni))]: $

$
  \begin{array}{ccccc}
    布局: & ( & (1,8)                        & , & (2048,256) & )                    & : & ((0,1),(2048,8)) & \xrightarrow{\text{切片}} & ((1,8)):((0,1)) \\
            &   & \underbrace{\phantom{(1,8)}} &   & \underbrace{\phantom{(2048,256)}} &   & \\
    坐标:  & ( & None                         & , & (mi, ni)   & )                    &   &
  \end{array}
$

其中None 表示取整个块，(mi, ni) 是 thread 在第二个模式中的坐标。然后可以通过 `gA[(None, (mi, ni))].load()` 方法将 tensor 数据加载到向量中。这等效于

```python
v0 = gA[(0, (mi, ni))]   # => mA[(mi, ni * 8 + 0)]
v1 = gA[(1, (mi, ni))]   # => mA[(mi, ni * 8 + 1)]
v2 = gA[(2, (mi, ni))]   # => mA[(mi, ni * 8 + 2)]
v3 = gA[(3, (mi, ni))]   # => mA[(mi, ni * 8 + 3)]
v4 = gA[(4, (mi, ni))]   # => mA[(mi, ni * 8 + 4)]
v5 = gA[(5, (mi, ni))]   # => mA[(mi, ni * 8 + 5)]
v6 = gA[(6, (mi, ni))]   # => mA[(mi, ni * 8 + 6)]
v7 = gA[(7, (mi, ni))]   # => mA[(mi, ni * 8 + 7)]
```


用一个 4x8 的小矩阵走一遍完整流程：

```python
mA: (4, 8) : (8, 1)    # 4x8 行主序矩阵

gA = cute.zipped_divide(mA, tiler=(1, 4))
# gA: ((1,4), (4,2)) : ((0,1), (8,4))
#      ^^^^   ^^^^
#      块内    块数
```

原始矩阵在内存中的布局（每行 8 个元素，`|` 表示 tiler 边界）：

```
行0: [ 0  1  2  3 | 4  5  6  7 ]
行1: [ 8  9 10 11 |12 13 14 15 ]
行2: [16 17 18 19 |20 21 22 23 ]
行3: [24 25 26 27 |28 29 30 31 ]
```

按 tiler=(1,4) 切分后，M 方向 4/1=4 个块，N 方向 8/4=2 个块，共 4x2=8 个块。

**thread_idx 到块坐标的映射**（对应 kernel 中的代码）：

```python
m, n = gA.shape[1]     # 第二个模式的 shape = (4, 2)，即 m=4, n=2
mi = thread_idx // n   # 行索引，慢速变化
ni = thread_idx % n    # 列索引，快速变化 → 保证 warp 内合并访存
```

| thread_idx | mi = idx//2 | ni = idx%2 | 块坐标 (mi,ni) | 处理的元素 |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | (0,0) | [0, 1, 2, 3] |
| 1 | 0 | 1 | (0,1) | [4, 5, 6, 7] |
| 2 | 1 | 0 | (1,0) | [8, 9, 10, 11] |
| 3 | 1 | 1 | (1,1) | [12, 13, 14, 15] |
| 4 | 2 | 0 | (2,0) | [16, 17, 18, 19] |
| 5 | 2 | 1 | (2,1) | [20, 21, 22, 23] |
| 6 | 3 | 0 | (3,0) | [24, 25, 26, 27] |
| 7 | 3 | 1 | (3,1) | [28, 29, 30, 31] |

**切片操作**以 thread 3（mi=1, ni=1）为例：

```python
a_val = gA[(None, (1, 1))].load()
# None  → 保留第一个模式，取块内全部 4 个元素
# (1,1) → 选中第二个模式的坐标，即行1、列4-7
# 结果：(1,4):(0,1) 的子 tensor，指向内存地址 [12, 13, 14, 15]
# .load() 用一条 64-bit 向量指令加载这 4 个元素
```

ni 用取模（快速变化维度）的原因：同一 warp 内相邻 thread（如 thread 0 和 1）的 ni 不同但 mi 相同，它们访问同一行内相邻的块，内存地址连续，保证了合并访存（coalesced access）。


为了引导编译器使用向量化加载/存储，我们必须告诉编译器假设传入指针的对齐方式。用户需要保证运行时的实际指针满足对齐限制。

```python
a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

### 使用对齐假设编译 kernel
compiled_func = cute.compile(vectorized_elementwise_add, a_, b_, c_)
```

值得注意的是，分区或分块的 tensor 可能由于子切片期间的偏移而具有不同的基指针对齐方式。

In [9]:
@cute.jit
def vectorized_elementwise_add(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    threads_per_block = 256

    gA = cute.zipped_divide(mA, (1, 8))
    gB = cute.zipped_divide(mB, (1, 8))
    gC = cute.zipped_divide(mC, (1, 8))

    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA}")
    print(f"[DSL INFO]   gB = {gB}")
    print(f"[DSL INFO]   gC = {gC}")

    vectorized_elementwise_add_kernel(gA, gB, gC).launch(
        grid=(cute.size(gC, mode=[1]) // threads_per_block, 1, 1),
        block=(threads_per_block, 1, 1),
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

compiled_func = cute.compile(vectorized_elementwise_add, a_, b_, c_)
compiled_func(a_, b_, c_)

# 验证正确性
torch.testing.assert_close(c, a + b)

[DSL INFO] Tiled Tensors:
[DSL INFO]   gA = tensor<ptr<f16, gmem, align<16>> o ((1,4),(16384,2048)):((0,1),(8192,4))>
[DSL INFO]   gB = tensor<ptr<f16, gmem, align<16>> o ((1,4),(16384,2048)):((0,1),(8192,4))>
[DSL INFO]   gC = tensor<ptr<f16, gmem, align<16>> o ((1,4),(16384,2048)):((0,1),(8192,4))>
[DSL INFO] sliced gA = tensor<ptr<f16, gmem, align<8>> o ((1,4)):((0,1))>
[DSL INFO] sliced gB = tensor<ptr<f16, gmem, align<8>> o ((1,4)):((0,1))>


In [10]:
benchmark(compiled_func, a_, b_, c_)

性能指标:
-------------------
Kernel 执行时间: 135.2678 us
内存吞吐量: 5953.42 GB/s


## 五、使用 TV 布局的逐元素操作

### 从手动映射到 TV 布局

回顾前面向量化 kernel 中手动完成的两步映射：

```
thread_idx  ──(mi=idx//n, ni=idx%n)──>  逻辑坐标 (mi, ni)  ──(tensor布局)──>  物理地址
```

这里混合了两个关注点：
- **thread 在块之间的分配**：哪个 thread 处理哪个块（`mi`、`ni` 的计算）
- **每个 thread 内的元素访问**：块内 8 个元素的向量化加载

当我们想调整向量化宽度或者改变 thread 的排列方式（比如让更多 thread 排在列方向），就需要同时修改 `mi`/`ni` 计算和 `tiler` 参数。

**TV 布局**把这两个关注点统一成一个布局对象，形式为 `(thread_domain, value_domain) : (thread_stride, value_stride)`：

- **thread_domain**：所有 thread 的索引空间。给定 thread_idx，TV 布局直接告诉你该 thread 对应的逻辑坐标起始位置
- **value_domain**：单个 thread 内的 value 索引空间。给定 value_idx（0 到 3），TV 布局告诉你这是块内第几个元素

用前面 4x8 矩阵（tiler=(1,4)）的例子对比。以 thread 3 为例：

**手动映射**需要两步：
1. 算块坐标：`mi=3//2=1, ni=3%2=1` → 块 (1,1)
2. 算原始坐标：第 0 个元素 → `mA[1, 1*4+0]` = `mA[1, 4]`，第 2 个元素 → `mA[1, 1*4+2]` = `mA[1, 6]`

**TV 布局**一步到位，直接从 (thread_idx, value_idx) 映射到原始 tensor 坐标：
- `tv_layout(3, 0)` → (1, 4)，即 thread 3 的第 0 个元素在 `mA[1, 4]`
- `tv_layout(3, 2)` → (1, 6)，即 thread 3 的第 2 个元素在 `mA[1, 6]`

| | 手动映射 | TV 布局 |
|---|---|---|
| thread 3, value 0 | 块坐标 (1,1) → `mA[1, 4]` | `tv_layout(3, 0)` → `mA[1, 4]` |
| thread 3, value 2 | 块坐标 (1,1) → `mA[1, 6]` | `tv_layout(3, 2)` → `mA[1, 6]` |
| 改变向量化宽度 | 修改 tiler、n、`%`/`//` | 只换一个 TV 布局对象 |

### 两级分块流程

使用 TV 布局的 kernel 采用两级分块：

**第一级：block 级别分块**（Host 端完成）

在 host 端用 `zipped_divide` 将整个 tensor 分成 `(TileM, TileN)` 的子 tensor。每个 CUDA block 处理一个子 tensor。

**第二级：thread 级别分块**（Kernel 内完成）

在 kernel 内部，通过 TV 布局将 `(TileM, TileN)` 子 tensor 进一步分配给每个 thread。流程为：

```
gA[((None, None), bidx)]     # block 索引切片 → (TileM, TileN) 子 tensor
    ↓ composition(blkA, tv_layout)
tidfrgA                       # (tid, vid) → 物理地址
    ↓ tidfrgA[(tidx, None)]
thrA                          # 当前 thread 的数据视图
```

### TV 布局的构造

TV 布局由 thread 布局和 value 布局组合而成：

- **thread 布局** `(4, 64):(64, 1)`：256 个 thread 排列为 4 行 64 列，64 个 thread 在列方向（行维度）上连续排列以实现合并访存
- **value 布局** `(16, 8):(8, 1)`：每个 thread 读取 16 行、每行 8 个连续的元素

用 `cute.make_layout_tv(thr_layout, val_layout)` 组合后得到 `tiler_mn`（block 的分块大小）和 `tv_layout`（thread-value 到逻辑坐标的映射）。



### TV 布局详解：4x8 矩阵手算全过程

本节用一个 4x8 的小矩阵 mA: (4, 8):(8, 1)，数据类型bf16，逐步手算 TV 布局的构造和使用全过程。

---

**Host 侧（`elementwise_add` 函数）**

Host 侧的代码完成三件事：构造布局、生成 TV 映射、将矩阵分块后交给 kernel。

**1) 构造 thread 布局** —— 定义 thread 在 tile 中的排布

`make_ordered_layout(shape, order)` 生成 compact（紧凑）布局，**order 值越小的维度 stride 越小（越连续）**。

```python
make_ordered_layout((4, 4), order=(1, 0))  # 行主序: (4,4):(4,1)
make_ordered_layout((4, 4), order=(0, 1))  # 列主序: (4,4):(1,4)
```

```python
thr_layout = make_ordered_layout((2, 2), order=(1, 0))
# 结果: (2, 2) : (2, 1)    N维stride=1(最连续), M维stride=2
```

4 个 thread 排成 2x2 网格，`tid = row*2 + col`：

```
          col0   col1       (N方向/内存连续方向)
row0:  [  T0  |  T1  ]     tid=0*2+0=0, tid=0*2+1=1
row1:  [  T2  |  T3  ]     tid=1*2+0=2, tid=1*2+1=3
```

相邻 thread (T0, T1) 在 N 方向相邻，对行主序矩阵保证合并访存。

**2) 构造 value 布局** —— 定义每个 thread 内 value 的排布

```python
val_layout = make_ordered_layout((2, 4), order=(1, 0))
# 结果: (2, 4) : (4, 1)    N维stride=1(连续加载), M维stride=4
```

每个 thread 的 8 个 value 排成 2x4 网格，`vid = row*4 + col`：

```
          col0  col1  col2  col3     (N方向/连续加载)
row0:  [  V0  | V1  | V2  | V3  ]   vid = 0,1,2,3
row1:  [  V4  | V5  | V6  | V7  ]   vid = 4,5,6,7
```

**3) `make_layout_tv` 生成 TV 映射**

```python
tiler_mn, tv_layout = make_layout_tv(thr_layout, val_layout)
```

`make_layout_tv` 将 thread 布局和 value 布局组合，返回两个结果：

**tiler_mn** —— 每个 block 处理的 tile 大小：

```
TileM = thr_shape[0] * val_shape[0] = 2 * 2 = 4
TileN = thr_shape[1] * val_shape[1] = 2 * 4 = 8
tiler_mn = (4, 8)
```

**tv_layout** —— 从 `(tid, vid)` 到 tile 内逻辑坐标 `(row, col)` 的映射：

```python
tv_layout: ((2,2),(4,2)):((16,2),(4,1))
#           ^^^^^  ^^^^^
#          thread域 value域
```

效果是将 tile 切成 2x2 个矩形子块，每个 thread 拥有一个 2x4 大小的子块。

注意 shape 的"交换"：val_layout 是 `(2,4)` 但 tv_layout 的 value 域是 `(4,2)`。**规律：tv_layout 各域的 shape 是原始 layout shape 的逆序。** 这是 `make_layout_tv` 内部求逆映射的数学结果，不影响正确性——线性 thread 索引到 tile 坐标的映射不变。

TV 布局把 tile 切成了 **thr_shape[0] x thr_shape[1] 个规则的矩形子块**，每个 thread 拥有一个子块，子块大小为 **val_shape[0] x val_shape[1]**：

```
        ┌──── T0 的子块 ────┬──── T1 的子块 ────┐
行 0-1  │ V0 V1 V2 V3      │ V0 V1 V2 V3      │
        │ V4 V5 V6 V7      │ V4 V5 V6 V7      │
        ├──── T2 的子块 ────┼──── T3 的子块 ────┤
行 2-3  │ V0 V1 V2 V3      │ V0 V1 V2 V3      │
        │ V4 V5 V6 V7      │ V4 V5 V6 V7      │
        └───────────────────┴───────────────────┘
```

子块内部，**N 方向的连续元素**对应一次向量化加载。同一行内相邻 thread 的子块在 N 方向上相邻，保证了 warp 内的合并访存。

完整映射表（标注每个 tile 位置对应的 `T编号,V编号`）：

```
mA: (4, 8) 行主序, 元素编号 = row*8 + col

          col0     col1     col2     col3   | col4     col5     col6     col7
      +---------+---------+---------+-------++---------+---------+---------+---------+
row0  | T0, V0  | T0, V1  | T0, V2  | T0, V3 || T1, V0  | T1, V1  | T1, V2  | T1, V3  |
row1  | T0, V4  | T0, V5  | T0, V6  | T0, V7 || T1, V4  | T1, V5  | T1, V6  | T1, V7  |
      +---------+---------+---------+-------++---------+---------+---------+---------+
row2  | T2, V0  | T2, V1  | T2, V2  | T2, V3 || T3, V0  | T3, V1  | T3, V2  | T3, V3  |
row3  | T2, V4  | T2, V5  | T2, V6  | T2, V7 || T3, V4  | T3, V5  | T3, V6  | T3, V7  |
      +---------+---------+---------+-------++---------+---------+---------+---------+
```

每个 thread 负责的元素：

| Thread | 行范围 | 列范围 | 物理偏移 |
|--------|--------|--------|----------|
| T0 | 0-1 | 0-3 | 0,1,2,3,8,9,10,11 |
| T1 | 0-1 | 4-7 | 4,5,6,7,12,13,14,15 |
| T2 | 2-3 | 0-3 | 16,17,18,19,24,25,26,27 |
| T3 | 2-3 | 4-7 | 20,21,22,23,28,29,30,31 |

V0-V3 是第一行的 4 个连续元素（一次 8 字节向量化加载），V4-V7 是第二行的 4 个连续元素（又一次向量化加载）。

**4) `zipped_divide` 将矩阵分成 tile**

```python
gA = zipped_divide(mA, tiler_mn=(4, 8))
# gA: ((4, 8), (1, 1)) : ((8, 1), (0, 0))
#      ^^^^^   ^^^^^
#      tile内   block数(只有1个block)
```

4x8 矩阵刚好一个 tile（1 个 block），更大矩阵只是有更多 block。

此后 `gA`, `gB`, `gC` 和 `tv_layout` 被传入 kernel。

---

**Kernel 侧（`elementwise_add_kernel`）**

Kernel 内每个 thread 需要知道自己该读写哪些内存地址。四个步骤逐层缩小范围：整个矩阵 -> 当前 block 的 tile -> 当前 thread 的若干元素。

**第 1 步：选出当前 block 的 tile** —— "这个 block 负责矩阵的哪一块？"

```python
blkA = gA[((None, None), bidx)]
# bidx=0 时取第 0 个 tile
# blkA: (4, 8) : (8, 1)    shape 是 4行8列, stride 表示行主序
# blkA(row, col) = base + row*8 + col
```

整个矩阵被 Host 端的 `zipped_divide` 切成了多个 tile，`bidx` 选出当前 block 对应的那一块。`blkA` 是一个 4x8 的子 tensor，可以用 (row, col) 坐标访问。

**第 2 步：换一种索引方式** —— "thread X 的 value Y 对应 tile 中的哪个位置？"

```python
tidfrgA = composition(blkA, tv_layout)
# blkA:      (4, 8)          : (8, 1)             (row, col) -> 物理偏移
# tv_layout: ((2, 2), (4, 2)): ((16, 2), (4, 1))  (tid, vid) -> tile 内坐标
# 结果：
# tidfrgA:   ((2, 2), (4, 2)): ((4, 16), (1, 8))  (tid, vid) -> 物理偏移
#             ^^^^^^  ^^^^^^
#             thread域  value域     shape 继承自 tv_layout, stride 是物理偏移
```

这里有两套坐标系：
- **(row, col)**：tile 内的行列坐标，是矩阵的逻辑位置，`blkA` 用这个索引
- **(tid, vid)**：thread 编号和 value 编号，是 GPU 执行时每个 thread 天然知道的信息

问题是 `blkA` 用 (row, col) 索引，但 kernel 中每个 thread 只知道自己的 `tidx`，不知道该读哪行哪列。`composition` 把 tv_layout（描述了 tid/vid 到 row/col 的对应关系）嵌入到 `blkA` 中，让结果可以直接用 (tid, vid) 索引。

以 Thread 1（tid=1）的 Value 0（vid=0）为例：查上面的映射表，T1 的 V0 在 tile 坐标 **(0, 4)**（row0, col4），然后 `blkA(0, 4) = base + 0*8 + 4 = base + 4`。注意 (tid=1, vid=0) 和 tile 坐标 (0, 4) 是完全不同的两套坐标，composition 就是把"查 tv_layout"和"算物理地址"合成一步。

**第 3 步：每个 thread 取自己那份** —— "这个 thread 该加载哪些地址？"

```python
thrA = tidfrgA[(tidx, None)]
# tidx=2 时: tid_0=0, tid_1=1, 基址偏移 = 0*4 + 1*16 = 16
# thrA: (4, 2) : (1, 8)     8 个 value, stride=(1,8) 表示每 4 个连续
# thrA(vid) = base + 16 + vid_0*1 + vid_1*8
```

以 Thread 2 为例，`thrA` 就是 T2 那个 2x4 子块的 8 个元素地址：

| vid | tile (row, col) | 物理偏移 |
|-----|---------|---------|
| 0 | (2, 0) | 16 |
| 1 | (2, 1) | 17 |
| 2 | (2, 2) | 18 |
| 3 | (2, 3) | 19 |
| 4 | (3, 0) | 24 |
| 5 | (3, 1) | 25 |
| 6 | (3, 2) | 26 |
| 7 | (3, 3) | 27 |

**第 4 步：加载、计算、存储**

```python
thrC[None] = thrA.load() + thrB.load()
```

vid 0-3 在内存中连续（偏移 16,17,18,19），vid 4-7 也连续（偏移 24,25,26,27），编译器生成两条 64-bit 向量化加载指令。

---

**与 notebook 中实际参数的对应**

| | 4x8 简化例子 | notebook 实际值 |
|---|---|---|
| 矩阵大小 | (4, 8) | (16384, 8192) |
| thr_layout | (2, 2):(2, 1) | (4, 64):(64, 1) |
| val_layout | (2, 4):(4, 1) | (16, 8):(8, 1) |
| tiler_mn | (4, 8) | (64, 512) |
| thread 数 | 4 | 256 |
| 每 thread values | 8 | 128 |
| 向量化宽度(N方向) | 4 元素 = 8 字节 | 8 元素 = 16 字节 |
| block 数 | 1 | 256*16=4096 |


### Kernel 代码


In [11]:
@cute.kernel
def elementwise_add_kernel(
    gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor, tv_layout: cute.Layout
):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # --------------------------------
    # 切片获取 thread block 级别视图
    # --------------------------------
    blk_coord = ((None, None), bidx)

    # 逻辑坐标 -> 地址
    blkA = gA[blk_coord]  # (TileM, TileN) -> 物理地址
    blkB = gB[blk_coord]  # (TileM, TileN) -> 物理地址
    blkC = gC[blk_coord]  # (TileM, TileN) -> 物理地址

    # --------------------------------
    # 组合以实现 thread 索引和 value 索引到物理地址的映射
    # --------------------------------
    # blockA:    (TileM, TileN) -> 物理地址
    # tv_layout: (tid, vid)     -> (TileM, TileN)
    # tidfrgA = blkA o tv_layout
    # tidfrgA:   (tid, vid) -> 物理地址
    tidfrgA = cute.composition(blkA, tv_layout)
    tidfrgB = cute.composition(blkB, tv_layout)
    tidfrgC = cute.composition(blkC, tv_layout)

    print("与 TV 布局组合后:")
    print(f"  tidfrgA: {tidfrgA.type}")

    # --------------------------------
    # 切片获取 thread 级别视图
    # --------------------------------
    # `None` 表示切片整个每 thread 数据
    thr_coord = (tidx, None)
    # thr_coord = (tidx, cute.repeat_like(None, gA.shape[1]))

    # 为 thread 切片：vid -> 地址
    thrA = tidfrgA[thr_coord]  # (V) -> 物理地址
    thrB = tidfrgB[thr_coord]  # (V) -> 物理地址
    thrC = tidfrgC[thr_coord]  # (V) -> 物理地址

    thrC[None] = thrA.load() + thrB.load()

### Host 代码

为了通用化不同数据类型，value 布局先以字节为单位定义，再用 `recast_layout` 转换为元素单位：

```python
val_layout = cute.recast_layout(dtype.width, 8, bit_val_layout)      # 源类型位数：8, 目标类型位数：元素类型的位数
```

In [12]:
@cute.jit
def elementwise_add(
    mA: cute.Tensor,
    mB: cute.Tensor,
    mC: cute.Tensor,
):
    # mA 布局: (M, N):(N, 1)
    # TV 布局将 thread 和 value 索引映射到 (64, 512) 逻辑分块
    #  - 连续的 thread 索引映射到模式 1，因为输入布局在模式 1 上是连续的，以实现访存合并
    #  - 每个 thread 每行加载连续 16 字节，加载 16 行
    coalesced_ldst_bytes = 16

    # 编译时验证：期望所有输入 tensor 具有相同的元素类型
    assert all(t.element_type == mA.element_type for t in [mA, mB, mC])
    dtype = mA.element_type

    thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))  # (4,64):(64,1)
    val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))  # (16,16):(16,1)
    val_layout = cute.recast_layout(dtype.width, 8, val_layout)  # (16,8):(8,1)
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)  # (64, 512), ((64,4),(8,16)):((512,16),(64,1))

    print(f"[DSL INFO] Tiler: {tiler_mn}")
    print(f"[DSL INFO] TV Layout: {tv_layout}")

    gA = cute.zipped_divide(mA, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gB = cute.zipped_divide(mB, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gC = cute.zipped_divide(mC, tiler_mn)  # ((TileM, TileN), (RestM, RestN))

    print("分块后的输入 Tensor：")
    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA.type}")
    print(f"[DSL INFO]   gB = {gB.type}")
    print(f"[DSL INFO]   gC = {gC.type}")

    # 异步启动 kernel
    # 也可以指定异步 token 作为依赖项
    elementwise_add_kernel(gA, gB, gC, tv_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

elementwise_add_ = cute.compile(elementwise_add, a_, b_, c_)
elementwise_add_(a_, b_, c_)

# 验证正确性
torch.testing.assert_close(c, a + b)

[DSL INFO] Tiler: (64, 512)
[DSL INFO] TV Layout: ((64,4),(8,16)):((512,16),(64,1))
分块后的输入 Tensor：
[DSL INFO] Tiled Tensors:
[DSL INFO]   gA = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
[DSL INFO]   gB = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
[DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
与 TV 布局组合后:
  tidfrgA: !cute.memref<f16, gmem, align<16>, "((64,4),(8,16)):((8,131072),(1,8192))">


### TV 布局可视化

要可视化 TV 布局，我们可以先安装 *`cute-viz`*

```
pip install -U git+https://github.com/NTT123/cute-viz.git
```

In [25]:
try:
    from cute_viz import display_tv_layout

    @cute.jit
    def visualize():
        # 4x8 小矩阵的 TV 布局可视化
        tv_layout = cute.make_layout(((2, 2), (4, 2)), stride=((16, 2), (4, 1)))
        display_tv_layout(tv_layout, (4, 8))

        # 原始 16x256 tile 的 TV 布局可视化（对应 notebook 中 256x512 矩阵）
        # tv_layout_large = cute.make_layout(((32, 4), (8, 4)), stride=((128, 4), (16, 1)))
        # display_tv_layout(tv_layout_large, (16, 256))

    visualize()
except (ImportError, Exception):
    pass

In [14]:
benchmark(elementwise_add_, a_, b_, c_)

性能指标:
-------------------
Kernel 执行时间: 124.5632 us
内存吞吐量: 6465.04 GB/s


### 重映射/转置 block 索引

由于本示例中的 tensor 是行优先的，我们可能希望 thread block 尽可能加载连续内存。

我们可以应用简单的 thread block 重映射来转置 thread block 索引的映射为行优先顺序。`cute.composition(gA, (None, remap_block))` 只对分块布局的第 2 个模式应用转置，但保持第 1 个模式不变。

```python
    remap_block = cute.make_ordered_layout(
        cute.select(gA.shape[1], mode=[1, 0]), order=(1, 0)
    )
    gA = cute.composition(gA, (None, remap_block))
    gB = cute.composition(gB, (None, remap_block))
    gC = cute.composition(gC, (None, remap_block))
```

In [15]:
@cute.jit
def elementwise_add(
    mA: cute.Tensor,
    mB: cute.Tensor,
    mC: cute.Tensor,
):
    # mA 布局: (M, N):(N, 1)
    # TV 布局将 thread 和 value 索引映射到 (64, 512) 逻辑分块
    #  - 连续的 thread 索引映射到模式 1，因为输入布局在模式 1 上是连续的
    #     以实现合并加载-存储
    #  - 每个 thread 每行加载连续 16 字节，加载 16 行
    coalesced_ldst_bytes = 16

    # 编译时验证：期望所有输入 tensor 具有相同的元素类型
    assert all(t.element_type == mA.element_type for t in [mA, mB, mC])
    dtype = mA.element_type

    thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))
    val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))
    val_layout = cute.recast_layout(dtype.width, 8, val_layout)
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    print(f"[DSL INFO] Tiler: {tiler_mn}")
    print(f"[DSL INFO] TV Layout: {tv_layout}")

    gA = cute.zipped_divide(mA, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gB = cute.zipped_divide(mB, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gC = cute.zipped_divide(mC, tiler_mn)  # ((TileM, TileN), (RestM, RestN))

    # (RestM, RestN) -> (RestN, RestM)
    remap_block = cute.make_ordered_layout(
        cute.select(gA.shape[1], mode=[1, 0]), order=(1, 0)
    )
    gA = cute.composition(gA, (None, remap_block))
    gB = cute.composition(gB, (None, remap_block))
    gC = cute.composition(gC, (None, remap_block))

    print("分块后的输入 Tensor：")
    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA.type}")
    print(f"[DSL INFO]   gB = {gB.type}")
    print(f"[DSL INFO]   gC = {gC.type}")

    # 异步启动 kernel
    # 也可以指定异步 token 作为依赖项
    elementwise_add_kernel(gA, gB, gC, tv_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

elementwise_add_ = cute.compile(elementwise_add, a_, b_, c_)
elementwise_add_(a_, b_, c_)

# 验证正确性
torch.testing.assert_close(c, a + b)

[DSL INFO] Tiler: (64, 512)
[DSL INFO] TV Layout: ((64,4),(8,16)):((512,16),(64,1))
分块后的输入 Tensor：
[DSL INFO] Tiled Tensors:
[DSL INFO]   gA = !cute.memref<f16, gmem, align<16>, "((64,512),(16,256)):((8192,1),(512,524288))">
[DSL INFO]   gB = !cute.memref<f16, gmem, align<16>, "((64,512),(16,256)):((8192,1),(512,524288))">
[DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "((64,512),(16,256)):((8192,1),(512,524288))">
与 TV 布局组合后:
  tidfrgA: !cute.memref<f16, gmem, align<16>, "((64,4),(8,16)):((8,131072),(1,8192))">


In [16]:
benchmark(compiled_func, a_, b_, c_)

性能指标:
-------------------
Kernel 执行时间: 135.1965 us
内存吞吐量: 5956.56 GB/s


## 六、使用 Lambda 函数

CuTe DSL 构建在 Python 之上。它可以利用 Python 实现元编程来生成灵活的 kernel。例如，我们可以编写接受自定义二元操作的 kernel 模板，为任意二元操作生成 kernel。


```python
@cute.jit
def elementwise_apply(
    op: cutlass.Constexpr,
    inputs,
    result: cute.Tensor
):
    ...

```

In [17]:
@cute.kernel
def elementwise_apply_kernel(
    op: cutlass.Constexpr,
    mInputs: List[cute.Tensor],
    mC: cute.Tensor,
    cC: cute.Tensor,  # coordinate tensor
    shape: cute.Shape,
    tv_layout: cute.Layout,  # (tid, vid) -> logic coord
):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    ###############################################################################
    # 切片到 thread block 的本地分块
    ###############################################################################
    blk_crd = ((None, None), bidx)

    # 利用 DSL 的元编程能力为每个输入切片 tensor
    # 下面对输入 tensor 的所有 for 循环在编译时会自动完全展开
    # 逻辑坐标 -> 内存地址
    gInputs = [t[blk_crd] for t in mInputs]  # (TileM, TileN)
    gC = mC[blk_crd]  # (TileM, TileN)
    gCrd = cC[blk_crd]  # (TileM, TileN)

    print("[DSL INFO] Sliced Tensors per thread block:")
    for i in cutlass.range_constexpr(len(gInputs)):
        print(f"[DSL INFO]   ctaInputs{i} = {gInputs[i].type}")
    print(f"[DSL INFO]   gC = {gC.type}")
    print(f"[DSL INFO]   gCrd = {gCrd.type}")

    ###############################################################################
    # 与 thread block TV 布局组合，将 thread 和 value 索引映射到内存地址
    ###############################################################################
    # (tid, vid) -> 内存地址
    tidfrgInputs = [cute.composition(t, tv_layout) for t in gInputs]
    tidfrgC = cute.composition(gC, tv_layout)
    tidfrgCrd = cute.composition(gCrd, tv_layout)

    # 用 None 重复以匹配 vid，移除布局层次结构
    thr_crd = (tidx, cute.repeat_like(None, tidfrgInputs[0][1]))

    ###############################################################################
    # 切片到 thread 的本地分块
    ###############################################################################
    # vid -> 地址
    thrInputs = [t[thr_crd] for t in tidfrgInputs]  # (V)
    thrC = tidfrgC[thr_crd]  # (V)
    thrCrd = tidfrgCrd[thr_crd]

    print("[DSL INFO] Sliced Tensors per thread:")
    for i in cutlass.range_constexpr(len(thrInputs)):
        print(f"[DSL INFO]   thrInputs{i} = {thrInputs[i].type}")
    print(f"[DSL INFO]   thrC = {thrC.type}")
    print(f"[DSL INFO]   thrCrd = {thrCrd.type}")

    ###############################################################################
    # 计算边界检查的谓词
    ###############################################################################
    frgPred = cute.make_fragment(thrCrd.shape, cutlass.Boolean)
    print(f"[DSL INFO]   frgPred = {frgPred.type}")

    for i in cutlass.range_constexpr(cute.size(frgPred)):
        frgPred[i] = cute.elem_less(thrCrd[i], shape)

    # if tidx == 0 and bidx == 0:
    #     cute.print_tensor(frgPred)

    ##########################################################
    # 加载数据并计算结果
    ##########################################################

    # 在使用前加载数据。编译器会优化复制和加载
    # 操作，将某些内存加载/存储转换为寄存器使用。
    result = op(*[thrInput.load() for thrInput in thrInputs])
    thrC.store(result)


@cute.jit
def elementwise_apply(op: cutlass.Constexpr, inputs, result: cute.Tensor):
    # 使用 128 位（16B）加载作为 val_layout 的规范形式，然后重新转换为目标元素类型
    coalesced_ldst_bytes = 16

    # 编译时验证：期望所有输入 tensor 具有相同的元素类型
    assert all(t.element_type == inputs[0].element_type for t in inputs)
    dtype = inputs[0].element_type

    thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))
    val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))
    val_layout = cute.recast_layout(dtype.width, 8, val_layout)
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    mInputs = [cute.zipped_divide(input, tiler_mn) for input in inputs]
    mC = cute.zipped_divide(result, tiler_mn)  # ((TileM, TileN), (RestM, RestN))

    # (RestM, RestN) -> (RestN, RestM)
    remap_block = cute.make_ordered_layout(
        cute.select(mInputs[0].shape[1], mode=[1, 0]), order=(1, 0)
    )
    for i, t in enumerate(mInputs):
        mInputs[i] = cute.composition(t, (None, remap_block))

    mC = cute.composition(mC, (None, remap_block))

    idC = cute.make_identity_tensor(result.shape)
    cC = cute.zipped_divide(idC, tiler=tiler_mn)

    # 异步启动 kernel
    # 将输入 tensor 分组到一个列表中作为单个参数
    elementwise_apply_kernel(op, mInputs, mC, cC, result.shape, tv_layout).launch(
        grid=[cute.size(mC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

In [18]:
from operator import mul

elementwise_apply(mul, [a_, b_], c_)

# 验证正确性
torch.testing.assert_close(c, mul(a, b))

[DSL INFO] Sliced Tensors per thread block:
[DSL INFO]   ctaInputs0 = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   ctaInputs1 = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   gCrd = !cute.coord_tensor<"(?{div=64},?{div=512})", "(64,512):(1@0,1@1)">
[DSL INFO] Sliced Tensors per thread:
[DSL INFO]   thrInputs0 = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrInputs1 = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrC = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrCrd = !cute.coord_tensor<"(?{div=16},?{div=8})", "((8,16)):((1@1,1@0))">
[DSL INFO]   frgPred = !cute.memref<i8, rmem, align<32>, "((8,16)):((1,8))">


### 使用自定义函数

自定义操作符可以更复杂。例如，下面是一个执行乘法后接 ReLU 的函数：

In [19]:
def mul_relu(a, b):
    tmp = a * b
    return cute.where(tmp > 0, tmp, cute.full_like(tmp, 0))


# 由于我们在自定义操作中使用了 cute.where，我们需要创建另一个 relu 函数
def mul_relu_ref(a, b):
    tmp = a * b
    return torch.relu(tmp)


elementwise_apply(mul_relu, [a_, b_], c_)

# 验证正确性
torch.testing.assert_close(c, mul_relu_ref(a, b))

[DSL INFO] Sliced Tensors per thread block:
[DSL INFO]   ctaInputs0 = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   ctaInputs1 = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "(64,512):(8192,1)">
[DSL INFO]   gCrd = !cute.coord_tensor<"(?{div=64},?{div=512})", "(64,512):(1@0,1@1)">
[DSL INFO] Sliced Tensors per thread:
[DSL INFO]   thrInputs0 = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrInputs1 = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrC = !cute.memref<f16, gmem, align<16>, "((8,16)):((1,8192))">
[DSL INFO]   thrCrd = !cute.coord_tensor<"(?{div=16},?{div=8})", "((8,16)):((1@1,1@0))">
[DSL INFO]   frgPred = !cute.memref<i8, rmem, align<32>, "((8,16)):((1,8))">



## 七、Autotune 和 Benchmark

In [20]:
import torch

import cutlass
import cutlass.cute as cute
import cutlass.cute.testing as testing
import cutlass.torch as cutlass_torch


CuTe DSL 提供了 autotune 和 benchmark 工具，帮助用户评估和优化 kernel 性能。本 notebook 演示如何使用这些工具。



### Autotune

我们为用户提供了两种 autotune 工具：`autotune.jit` 装饰器和 `tune` 函数。前者作为装饰器使用，放在 `@cute.jit` 之上；后者作为独立函数使用。


我们以 `elementwise_add_kernel` 为例。编写完 jit host 函数和 kernel 后，可以在 jit host 函数上方添加 `@autotune_jit` 装饰器来启用 autotune。
```python
@testing.autotune_jit(
    params_dict={"copy_bits": [64, 128]},
    update_on_change=["M", "N"],
    warmup_iterations=100,
    iterations=100,
)
```

`autotune_jit` 装饰器提供了几个参数来控制自动调优过程：

- params_dict：包含待调优参数及其可能值的字典
- update_on_change：当这些参数值发生变化时触发重新调优的参数名列表
- warmup_iterations：计时前的预热迭代次数
- iterations：每个参数组合的计时迭代次数


In [21]:
@cute.kernel
def elementwise_add_kernel(
    gA: cute.Tensor,
    gB: cute.Tensor,
    gC: cute.Tensor,
    cC: cute.Tensor,  # 坐标 tensor
    shape: cute.Shape,
    thr_layout: cute.Layout,
    val_layout: cute.Layout,
):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # 为 CTA 切片
    # 逻辑 id -> 地址
    blk_coord = ((None, None), bidx)
    blkA = gA[blk_coord]  # (TileM,TileN)
    blkB = gB[blk_coord]  # (TileM,TileN)
    blkC = gC[blk_coord]  # (TileM,TileN)
    blkCrd = cC[blk_coord]  # (TileM, TileN)

    # 声明后续用于内存拷贝的 atom
    copy_atom_load = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), gA.element_type)
    copy_atom_store = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), gC.element_type)

    tiled_copy_A = cute.make_tiled_copy_tv(copy_atom_load, thr_layout, val_layout)
    tiled_copy_B = cute.make_tiled_copy_tv(copy_atom_load, thr_layout, val_layout)
    tiled_copy_C = cute.make_tiled_copy_tv(copy_atom_store, thr_layout, val_layout)

    thr_copy_A = tiled_copy_A.get_slice(tidx)
    thr_copy_B = tiled_copy_B.get_slice(tidx)
    thr_copy_C = tiled_copy_C.get_slice(tidx)

    thrA = thr_copy_A.partition_S(blkA)
    thrB = thr_copy_B.partition_S(blkB)
    thrC = thr_copy_C.partition_S(blkC)

    # 为 gmem->rmem 分配 fragment
    frgA = cute.make_fragment_like(thrA)
    frgB = cute.make_fragment_like(thrB)
    frgC = cute.make_fragment_like(thrC)

    thrCrd = thr_copy_C.partition_S(blkCrd)
    frgPred = cute.make_rmem_tensor(thrCrd.shape, cutlass.Boolean)

    for i in range(0, cute.size(frgPred), 1):
        val = cute.elem_less(thrCrd[i], shape)
        frgPred[i] = val

    ##########################################################
    # 将数据移动到寄存器地址空间
    ##########################################################

    cute.copy(copy_atom_load, thrA, frgA, pred=frgPred)
    cute.copy(copy_atom_load, thrB, frgB, pred=frgPred)

    # 在使用前加载数据。编译器会优化复制和加载
    # 操作，将某些内存加载/存储转换为寄存器使用。
    result = frgA.load() + frgB.load()

    # 将结果保存回寄存器。这里复用 b 的寄存器。
    frgC.store(result)

    # 将结果拷贝回 c
    cute.copy(copy_atom_store, frgC, thrC, pred=frgPred)


@testing.autotune_jit(
    params_dict={"copy_bits": [64, 128]},
    update_on_change=["M", "N"],
    warmup_iterations=100,
    iterations=100,
)
@cute.jit
def elementwise_add_autotune(mA, mB, mC, M, N, copy_bits: cutlass.Constexpr = 128):
    dtype = mA.element_type
    vector_size = copy_bits // dtype.width

    thr_layout = cute.make_ordered_layout((4, 32), order=(1, 0))
    val_layout = cute.make_ordered_layout((4, vector_size), order=(1, 0))
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    gA = cute.zipped_divide(mA, tiler_mn)  # ((TileM,TileN),(RestM,RestN))
    gB = cute.zipped_divide(mB, tiler_mn)  # ((TileM,TileN),(RestM,RestN))
    gC = cute.zipped_divide(mC, tiler_mn)  # ((TileM,TileN),(RestM,RestN))
    idC = cute.make_identity_tensor(mC.shape)
    cC = cute.zipped_divide(idC, tiler=tiler_mn)

    elementwise_add_kernel(gA, gB, gC, cC, mC.shape, thr_layout, val_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )

当我们运行 jit 函数 `elementwise_add_autotune` 时，CuTe DSL 会通过循环遍历指定的配置来帮助我们调优 kernel，并使用最佳配置运行 kernel。




In [22]:

M, N = 1024, 1024
dtype = cutlass.Float32
skip_ref_check = False

print(f"\nRunning Elementwise Add test with:")
print(f"Tensor dimensions: [{M}, {N}]")
print(f"Input and Output Data type: {dtype}")

torch_dtype = cutlass_torch.dtype(dtype)

a = torch.randn(M, N, device=torch.device("cuda"), dtype=torch_dtype)
b = torch.randn(M, N, device=torch.device("cuda"), dtype=torch_dtype)

c = torch.zeros_like(a)

print(f"Input tensor shapes:")
print(f"a: {a.shape}, dtype: {a.dtype}")
print(f"b: {b.shape}, dtype: {b.dtype}")
print(f"c: {c.shape}, dtype: {c.dtype}\n")

elementwise_add_autotune(a, b, c, M, N)

if not skip_ref_check:
    print("Verifying results for autotuned function ...")
    torch.testing.assert_close(a + b, c)
    print("Results verified successfully!")


Running Elementwise Add test with:
Tensor dimensions: [1024, 1024]
Input and Output Data type: Float32
Input tensor shapes:
a: torch.Size([1024, 1024]), dtype: torch.float32
b: torch.Size([1024, 1024]), dtype: torch.float32
c: torch.Size([1024, 1024]), dtype: torch.float32

Verifying results for autotuned function ...
Results verified successfully!


输出如下：

```
Running Elementwise Add test with:
Tensor dimensions: [1024, 1024]
Input and Output Data type: Float32
Input tensor shapes:
a: torch.Size([1024, 1024]), dtype: torch.float32
b: torch.Size([1024, 1024]), dtype: torch.float32
c: torch.Size([1024, 1024]), dtype: torch.float32
Verifying results for autotuned function ...
Results verified successfully!
```


要详细监控自动调优过程，可以通过设置环境变量 `CUTE_DSL_LOG_AUTOTUNE` 来启用日志记录。
```shell
export CUTE_DSL_LOG_AUTOTUNE=1
```
这将显示全面的信息，包括：
- 正在评估的每个配置及其相应的执行时间
- 选择的最优配置
- 调优花费的总时间
- 缓存命中/未命中统计


以下是显示不同配置自动调优过程的示例输出：
```python
2025-07-23 06:17:03,978 - cutlass.cute.testing_Autotune - INFO - Tuning configuration: {'copy_bits': 64}
2025-07-23 06:17:04,519 - cutlass.cute.testing_Autotune - INFO -    Execution time: 0.010857919985428453 us
2025-07-23 06:17:04,519 - cutlass.cute.testing_Autotune - INFO - Tuning configuration: {'copy_bits': 128}
2025-07-23 06:17:04,683 - cutlass.cute.testing_Autotune - INFO -    Execution time: 0.011117440033704042 us
2025-07-23 06:17:04,683 - cutlass.cute.testing_Autotune - INFO - Best configuration: {'copy_bits': 64}, execution time: 0.010857919985428453 us
2025-07-23 06:17:04,683 - cutlass.cute.testing_Autotune - INFO - Total tuning time: 0.7053244113922119 s
...
2025-07-23 06:17:04,700 - cutlass.cute.testing_Autotune - INFO - Using cached best configuration: {'copy_bits': 64}
```



我们还提供了一个 `tune` 函数。`tune` 函数的接口如下：

```python
def tune(
    func: Callable[[Any], Callable[[], Any]],
    params_dict: Dict[str, List[Any]] = None,
    kernel_arguments: JitArguments = JitArguments(),
    warmup_iterations=10,
    iterations=100,
    stream: Optional[cuda_driver.CUstream] = None,
) -> Dict[str, Any]:
```

`tune` 函数接受以下参数：

- func：一个可调用对象，接受配置参数并返回一个 kernel 函数
- params_dict：将参数名映射到待调优可能值列表的字典
- kernel_arguments：传递给 kernel 进行调优的参数
- warmup_iterations：计时前的预热迭代次数（默认：10）
- iterations：每个配置的计时迭代次数（默认：100）
- stream：用于执行的可选 CUDA stream。默认为默认 CUDA stream。stream 参数必须与传递给 kernel 的 stream 匹配，不匹配将导致错误。

它返回一个包含找到的最佳 kernel 配置的字典。


以下是使用 `tune` 函数的示例：

1. 首先从 `elementwise_add_autotune` 函数中移除 `@testing.autotune_jit` 装饰器：
    ```python
    @testing.autotune_jit(
        params_dict={"copy_bits": [64, 128]},
        update_on_change=["M", "N"], 
        warmup_iterations=100,
        iterations=100,
    )
    ```

 2. 定义一个 `tune_func`：
    - 接受输入 tensor（a, b, c）、维度（M, N）和调优参数 copy_bits
    - 使用 `cute.compile()` 编译 `elementwise_add_autotune` 函数
    - 返回一个执行编译后 kernel 的 lambda 函数

 3. 将 `tune_func` 传递给 `testing.tune` 函数，同时包含：
    - 要探索的参数空间（copy_bits 值）
    - 包装在 JitArguments 中的 kernel 参数
    - `tune` 函数将自动找到最优参数


In [23]:
def tune_func(a, b, c, M, N, copy_bits=128):
    compiled_func = cute.compile(elementwise_add_autotune, a, b, c, M, N, copy_bits=128)
    return lambda: compiled_func(a, b, c, M, N)

params = testing.tune(
    tune_func,
    params_dict={"copy_bits": [64, 128]},
    kernel_arguments=testing.JitArguments(a, b, c, M, N),
)
print(f"The best kernel configs found: {params}")

# 使用最佳配置运行 kernel
compiled_func = cute.compile(elementwise_add_autotune, a, b, c, M, N, **params)
compiled_func(a, b, c, M, N)
            

The best kernel configs found: {'copy_bits': 64}


0

### benchmark

在 CuTe DSL 中，benchmark 工具可用于测量 kernel 执行时间。benchmark 例程的接口如下：

```python
def benchmark(
    callable: Callable,
    *,
    warmup_iterations: int = 10,
    iterations: int = 100,
    stream: Optional[cuda_driver.CUstream] = None,
    kernel_arguments: Optional[JitArguments] = None,
    workspace_generator: Optional[Callable[[], JitArguments]] = None,
    workspace_count: int = 1,
    use_cuda_graphs: bool = False,
) -> float:
```

benchmark 工具公开了几个关键配置参数来控制性能分析行为：

- callable：要进行基准测试的函数
- warmup_iterations：控制测量开始前的初始预热迭代次数（默认：10）
- iterations：指定用于性能测量的迭代次数（默认：100）
- stream：指定在哪个 CUDA stream 上执行 kernel（默认：默认 stream）
- use_cuda_graphs：是否为可调用函数启用 CUDA graph 以最小化 kernel 启动开销（默认：False）
- workspace_generator：提供一个函数，每次迭代生成新的 kernel 参数以避免缓存效应
- workspace_count：确定在性能分析期间循环使用多少个不同的工作空间（默认：1）

进行基准测试时，可以配置几个关键参数：

1. 核心参数：
   - 要进行性能分析的函数（callable）
   - 测量前的预热迭代次数
   - 用于测量的性能分析迭代次数

2. Stream 配置：
   - 对于在非默认 stream 上运行的 kernel，必须指定 stream
   - stream 参数必须与传递给 kernel 的 stream 匹配，不匹配将导致错误

3. 缓存效应缓解：
   - 为防止 L2 缓存效应影响结果，可以循环使用多个工作空间
   - 这通过 workspace_count 和 workspace_generator 参数控制
   - 每个工作空间提供新的 kernel 参数

4. CUDA Graph 支持：
   - 支持测量不含 host 开销的 kernel 执行时间
   - 要求可调用函数使用 @cute.jit 装饰
   - 使用 graph 时必须使用非默认 CUDA stream

此函数返回可调用函数的执行时间，单位为微秒。由于 GPU 频率可能动态变化，我们可以固定 SM 和内存频率以获得更稳定和可重复的基准测试结果。这可以通过在运行基准测试前使用 nvidia-smi 设置 GPU 时钟来实现。接下来，让我们使用 benchmark 函数来获取上述 elementwise_add kernel 的执行时间。

In [24]:
def generate_kernel_arguments():
    a = torch.randn(
        M, N, device=torch.device("cuda"), dtype=torch_dtype
    )
    b = torch.randn(
        M, N, device=torch.device("cuda"), dtype=torch_dtype
    )

    c = torch.zeros_like(a)

    return testing.JitArguments(a, b, c, M, N)

avg_time_us = testing.benchmark(
    elementwise_add_autotune,
    workspace_generator=generate_kernel_arguments,
    workspace_count=10,
    warmup_iterations=10,
    iterations=100,
)

# Print execution results
print(
    f"Kernel execution time for cute.jit kernel with M={M}, N={N}: {avg_time_us / 1e3:.4f} ms"
)
print(
    f"Achieved memory throughput for M={M}, N={N}: {(3 * a.numel() * dtype.width // 8) / (avg_time_us / 1e6) / 1e9:.2f} GB/s"
)

Kernel execution time for cute.jit kernel with M=1024, N=1024: 0.0673 ms
Achieved memory throughput for M=1024, N=1024: 186.97 GB/s


运行代码后，我们将获得类似以下的输出：

```
Kernel execution time for cute.jit kernel with M=1024, N=1024: 0.0403 ms
Achieved memory throughput for M=1024, N=1024: 312.37 GB/s
```